# 风格因子暴露

从处理后的股票与指数数据计算八个风格因子，按日做截面处理，保存 `style_factor_exposure.parquet`。该结果同时用于 Alpha 中性化和联合收益模型。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

PANEL = PROJECT_ROOT / "data/processed/panel.parquet"
INDEX = PROJECT_ROOT / "data/processed/index.parquet"
STYLE_FACTOR_OUTPUT = PROJECT_ROOT / "src/factor_exposure/style_factor/style_factor_exposure.parquet"


## 读取并排序数据

股票按代码和日期排序，指数按日期排序。后续滞后与滚动窗口按市场交易日记录计算。


In [ ]:
index = pd.read_parquet(INDEX)
panel = pd.read_parquet(PANEL)

In [ ]:
panel = panel.sort_values(["ts_code", "trade_date"]).reset_index(drop=True)

index = index.sort_values("trade_date").reset_index(drop=True)


## 截面处理函数

`wz` 先按当日 1%/99% 分位数缩尾，再减均值、除以总体标准差。缺失值不参与截面统计。


In [ ]:
def wz(s):
    s = s.clip(lower=s.quantile(0.01), upper= s.quantile(0.99))
    s = (s - s.mean()) / s.std(ddof=0)
    return s

## 初始化结果

以日期和股票代码建立结果骨架，各风格因子分别计算并暂存，最后统一合并。下方保留各因子的公式与处理约定。


In [ ]:
style_factor_exposure = panel[['trade_date','ts_code']].copy()
style_factor_list = []

## 统一计算约定

$i$ 表示股票，$t$ 表示市场交易日。所有滞后及滚动窗口均按已排序的股票日频记录计算，包含停牌日，不跳过停牌记录。

### 收益率与缺失值

股票收益率读取 `panel.ret`，其上游计算口径为：

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
r_{i,t}=\frac{\widetilde C_{i,t}}{\widetilde C_{i,t-1}}-1
$$

$C$ 为后复权收盘价 `close_hfq`。按股票前向填充，不填充首次有效价格之前的数据。停牌期间收益率通常为 0，复牌日收益率相对停牌前最后有效收盘价计算。指数收益率为 $r_t^I=I_{C,t}/I_{C,t-1}-1$，读取 `index.ret`，不做填充。

### 指数权重

对于 200 日窗口、半衰期 $h$，统一定义：

$$
w_k^{(h)}=\frac{2^{-k/h}}{\sum_{j=0}^{199}2^{-j/h}},
\qquad k=0,\ldots,199
$$

$k=0$ 为所用窗口内最近的收益率，权重最大。Mom、Beta 使用 $h=50$，Vol1 使用 $h=30$。

### 截面处理及停牌日输出

以下各节公式描述原始值 $x_{i,t}^{\mathrm{raw}}$。对每个因子、每个交易日，先按 1% / 99% 分位数缩尾，再进行标准化：

$$
u_{i,t}=\operatorname{clip}\left(x_{i,t}^{\mathrm{raw}},q_{0.01,t},q_{0.99,t}\right),\qquad
z_{i,t}=\frac{u_{i,t}-\mu_t(u)}{\sigma_t(u)}
$$

$\sigma_t(u)$ 使用总体标准差（`ddof=0`）；截面统计忽略缺失值。流动性与价值因子在截面缩尾、标准化前，按 `suspend == True` 将当天值设为 `NaN`，不参与相应因子的当天截面统计。合并八个因子后，所有停牌日的最终风格暴露统一设为 `NaN`。其他风格因子的历史窗口及截面计算口径保持不变。


### Size：规模（`size`）

$$
\mathrm{Size}^{\mathrm{raw}}_{i,t}=\ln(\mathrm{MV}_{i,t})
$$

$\mathrm{MV}_{i,t}$ 为总市值 `total_mv`（万元）。先计算自然对数，再按当日截面缩尾、标准化。代码不对总市值进行填充。

In [ ]:
size_factor = panel[['trade_date','ts_code',"total_mv"]].copy()

In [ ]:
size_factor['size'] = np.log(size_factor["total_mv"])

size_factor["size"] = size_factor.groupby("trade_date")["size"].transform(wz)

size_factor = size_factor.drop(columns = ["total_mv"])

In [ ]:
style_factor_list.append(size_factor)

### Mom：动量（`mom`）

$$
\mathrm{Mom}^{\mathrm{raw}}_{i,t}
=\sum_{k=0}^{199}w_k^{(50)}\ln(1+r_{i,t-21-k})
$$

采用 200 日指数加权对数收益率，半衰期为 50 个交易日，滞后 21 个交易日；$k=0$ 对应 $t-21$，权重最大。代码为 `log_ret.shift(21).rolling(200, min_periods=200)`，需要从 $t-220$ 至 $t-21$ 的完整收益率窗口。原始值随后按当日截面缩尾、标准化。

In [ ]:
mom_factor = panel[
    ["trade_date", "ts_code", "ret"]
].copy()

mom_factor["log_ret"] = (
    mom_factor.groupby("ts_code")["ret"]
    .transform(lambda s: np.log1p(s))
)

In [ ]:
# 2. 设置指数权重
window = 200
lag = 21
half_life = 50

weights = 2.0 ** (
    -np.arange(window) / half_life
)
weights /= weights.sum()

def calc_rstr(log_return):
    # 排除最近21个交易日的收益
    x = log_return.shift(lag)

    # 对最近200个有效收益进行指数加权
    return x.rolling(
        window=window,
        min_periods=window,
    ).apply(
        lambda values: np.dot(values[::-1], weights),
        raw=True,
    )

mom_factor["mom"] = (
    mom_factor.groupby("ts_code")["log_ret"]
    .transform(calc_rstr)
)

mom_factor["mom"] =mom_factor.groupby('trade_date')["mom"].transform(wz)

mom_factor = mom_factor[
    ["trade_date", "ts_code", "mom"]
]

In [ ]:
style_factor_list.append(mom_factor)

### Vol1：收益率波动（`vol1`）

$$
\mu_{i,t}=\sum_{k=0}^{199}w_k^{(30)}r_{i,t-k}
$$

$$
\mathrm{Vol1}^{\mathrm{raw}}_{i,t}
=\sqrt{\sum_{k=0}^{199}w_k^{(30)}
\left(r_{i,t-k}-\mu_{i,t}\right)^2}
$$

使用包含当天的 200 日收益率窗口，半衰期为 30 个交易日。这是加权总体标准差，不作样本自由度修正。代码通过 $\sum w_kr^2-\mu^2$ 计算方差，并将浮点误差产生的微小负值截为 0；窗口含缺失收益率时结果为空。原始值随后按当日截面缩尾、标准化。

In [ ]:
vol_factor = panel[
    ["trade_date", "ts_code", "ret"]
].copy()


In [ ]:
window = 200
half_life = 30

# k = 0 表示当前日，k 越大表示越久远
k = np.arange(window, dtype=np.float64)

weights_recent_first = 2.0 ** (-k / half_life)
weights_recent_first /= weights_recent_first.sum()

# 滚动窗口中的数据顺序为：最早日 -> 当前日
# 因此将权重反转，使当前日获得最大权重
weights = weights_recent_first[::-1].copy()

# =========================
# 4. 计算 DASTD
# =========================
ret_all = vol_factor["ret"].to_numpy(dtype=np.float64)
dastd = np.full(len(vol_factor), np.nan)

for idx in vol_factor.groupby("ts_code", sort=False).indices.values():
    r = ret_all[idx]

    if len(r) < window:
        continue

    # 加权均值：sum(w * r)
    mu = np.correlate(r, weights, mode="valid")

    # 加权二阶矩：sum(w * r^2)
    second_moment = np.correlate(
        r * r, weights, mode="valid"
    )

    # 加权方差：
    # sum(w * (r - mu)^2) = sum(w * r^2) - mu^2
    variance = second_moment - mu**2

    # 避免浮点误差导致方差出现极小负数
    variance = np.maximum(variance, 0.0)

    # 前 window - 1 个位置对应的窗口不完整
    dastd[idx[window - 1:]] = np.sqrt(variance)

vol_factor["vol1"] = dastd

In [ ]:
vol_factor["vol1"] = vol_factor.groupby('trade_date')["vol1"].transform(wz)

### Vol2：分段累计收益极差（`vol2`）

$$
Z_{i,t}^{(j)}=\sum_{k=0}^{19}
\ln\left(1+r_{i,t-20(j-1)-k}\right),
\qquad j=1,\ldots,12
$$

$$
\mathrm{Vol2}^{\mathrm{raw}}_{i,t}
=\max_{1\le j\le12}Z_{i,t}^{(j)}
-\min_{1\le j\le12}Z_{i,t}^{(j)}
$$

从当天向前划分 12 个互不重叠的 20 日区间，每个区间计算累计对数收益率，再取这 12 个区间收益率的极差。总计需要 240 个交易日的收益率；任何区间不完整时结果为空（`skipna=False`）。这里使用分段收益率，不对各段继续累加。原始值随后按当日截面缩尾、标准化。

In [ ]:
vol_factor["log_ret"] = np.log1p(vol_factor["ret"])

In [ ]:
vol_factor["sum_logret_20"] = (
    vol_factor.groupby("ts_code", sort=False)["log_ret"]
    .transform(
        lambda s: s.rolling(20, min_periods=20).sum()
    )
)

for j in range(1, 13):
    vol_factor[f"Z{j}"] = (
        vol_factor.groupby("ts_code", sort=False)["sum_logret_20"]
        .shift(20 * (j - 1))
    )

z_cols = [f"Z{j}" for j in range(1, 13)]

vol_factor["vol2"] = (
    vol_factor[z_cols].max(axis=1, skipna=False)
    - vol_factor[z_cols].min(axis=1, skipna=False)
)

vol_factor["vol2"] = (
    vol_factor.groupby("trade_date")["vol2"].transform(wz)
)

In [ ]:
vol_factor = vol_factor[['trade_date', 'ts_code', 'vol1', 'vol2']]

In [ ]:
vol_factor.head()

In [ ]:
style_factor_list.append(vol_factor)

### Liq1：短期流动性（`liq1`）

$$
\mathrm{Liq1}^{\mathrm{raw}}_{i,t}
=\ln\left(\sum_{k=0}^{20}\widetilde{\mathrm{TO}}_{i,t-k}\right)
$$

换手率为 `turnover_rate`，其缺失值已在 `process.ipynb` 中统一填为 0，本笔记本不再填充。计算包含当天的 21 日累计换手率（`min_periods=21`）后取自然对数；累计值为 0 时结果为空。停牌日按 `suspend` 置为 `NaN`，再按当日截面缩尾、标准化。

In [ ]:
liq_factor = panel[['trade_date', 'ts_code', 'turnover_rate', 'suspend']].copy()

In [ ]:
# 计算过去 21 个交易日累计换手率
liq_factor["turnover_sum_21"] = (
    liq_factor.groupby("ts_code", sort=False)["turnover_rate"]
    .transform(lambda s: s.rolling(21, min_periods=21).sum())
)

# 计算 STOM
liq_factor["liq1"] = np.log(
    liq_factor["turnover_sum_21"].replace(0, np.nan)
)

liq_factor.loc[liq_factor['suspend'] == True, 'liq1'] = np.nan

liq_factor["liq1"] = liq_factor.groupby('trade_date')["liq1"].transform(wz)

### Liq2：长期流动性（`liq2`）

$$
\mathrm{Liq2}^{\mathrm{raw}}_{i,t}
=\ln\left(\sum_{k=0}^{199}\widetilde{\mathrm{TO}}_{i,t-k}\right)
$$

换手率为 `turnover_rate`，其缺失值已在 `process.ipynb` 中统一填为 0，本笔记本不再填充。计算包含当天的 200 日累计换手率（`min_periods=200`）后取自然对数；累计值为 0 时结果为空。停牌日按 `suspend` 置为 `NaN`，再按当日截面缩尾、标准化。

In [ ]:
liq_factor["turnover_sum_200"] = (
    liq_factor.groupby("ts_code", sort=False)["turnover_rate"]
    .transform(lambda s: s.rolling(200, min_periods=200).sum())
)

liq_factor["liq2"] = np.log(
    liq_factor["turnover_sum_200"].replace(0, np.nan)
)

liq_factor.loc[liq_factor['suspend'] == True, 'liq2'] = np.nan

liq_factor["liq2"] = liq_factor.groupby('trade_date')["liq2"].transform(wz)

In [ ]:
liq_factor = liq_factor[['trade_date','ts_code', 'liq1', 'liq2']]

In [ ]:
liq_factor.head()

In [ ]:
style_factor_list.append(liq_factor)

### Beta：市场敏感度（`beta`）

$$
\overline r_{i,t}=\sum_{k=0}^{199}w_k^{(50)}r_{i,t-k},
\qquad
\overline r_t^{I}=\sum_{k=0}^{199}w_k^{(50)}r_{t-k}^{I}
$$

$$
\mathrm{Beta}^{\mathrm{raw}}_{i,t}
=\frac{
\sum_{k=0}^{199}w_k^{(50)}
(r_{i,t-k}-\overline r_{i,t})
(r_{t-k}^{I}-\overline r_t^{I})
}{
\sum_{k=0}^{199}w_k^{(50)}
(r_{t-k}^{I}-\overline r_t^{I})^2
}
$$

采用包含当天的 200 日配对收益率窗口，半衰期为 50 个交易日。股票收益率读取 `panel.ret`，指数收益率读取 `index.ret`，按 `trade_date` 对齐。窗口含缺失收益率，或指数加权方差为 0 时结果为空。原始值随后按当日截面缩尾、标准化。

In [ ]:
beta_factor = panel[['trade_date','ts_code',"ret"]].copy()
idx = index[['trade_date',"close", 'ret']].copy()

In [ ]:
# 按交易日期匹配上证 A 股指数收益率
idx_ret = idx.set_index("trade_date")["ret"]
beta_factor["idx_ret"] = beta_factor["trade_date"].map(idx_ret)

# =========================
# 2. 构造指数加权权重
# =========================
window = 200
half_life = 50

k = np.arange(window, dtype=np.float64)

# k=0 对应当前交易日，权重最高
weights = 2.0 ** (-k / half_life)
weights /= weights.sum()


# =========================
# 3. 计算滚动加权 Beta
# =========================
ret_i_all = beta_factor["ret"].to_numpy(dtype=np.float64)
ret_m_all = beta_factor["idx_ret"].to_numpy(dtype=np.float64)

beta_values = np.full(len(beta_factor), np.nan)

for pos in beta_factor.groupby(
    "ts_code", sort=False
).indices.values():

    ri = ret_i_all[pos]
    rm = ret_m_all[pos]

    if len(pos) < window:
        continue

    # 加权均值
    # np.convolve 会反转卷积核，因此直接使用 weights
    # 即可使最新交易日对应最高权重
    mu_i = np.convolve(
        ri, weights, mode="valid"
    )

    mu_m = np.convolve(
        rm, weights, mode="valid"
    )

    # 加权二阶交叉矩
    cross_moment = np.convolve(
        ri * rm, weights, mode="valid"
    )

    # 指数收益率的加权二阶矩
    market_second_moment = np.convolve(
        rm * rm, weights, mode="valid"
    )

    # 加权协方差与加权方差
    cov_im = cross_moment - mu_i * mu_m

    var_m = market_second_moment - mu_m**2

    # 避免浮点误差造成极小负方差
    var_m = np.maximum(var_m, 0.0)

    # Beta = 加权协方差 / 加权市场方差
    beta = np.divide(
        cov_im,
        var_m,
        out=np.full_like(cov_im, np.nan),
        where=var_m > 0
    )

    # 前 199 个位置没有完整的 200 日窗口
    beta_values[pos[window - 1:]] = beta

beta_factor["beta"] = beta_values


beta_factor["beta"] = beta_factor.groupby('trade_date')["beta"].transform(wz)

In [ ]:
beta_factor = beta_factor[['ts_code','trade_date','beta']]

In [ ]:
style_factor_list.append(beta_factor)

### Value：价值（`value`）

$$
\mathrm{Value}^{\mathrm{raw}}_{i,t}=\frac{1}{PB_{i,t}}.
$$

采用账面市值比，数值越高表示相同市值对应更多账面净资产。`pb` 缺失或不大于零时结果为空，不填充。当天停牌先置为 `NaN`，再按 `trade_date` 执行 1%/99% 缩尾及总体标准差标准化，不参与停牌日截面统计。无需额外历史窗口；原始值当日 `pb` 有效即可计算，截面标准化还要求有效样本标准差非零。


In [ ]:
value_factor = panel[['trade_date', 'ts_code', 'pb', 'suspend']].copy()
value_factor['value'] = 1.0 / value_factor['pb'].where(value_factor['pb'].gt(0))
value_factor.loc[value_factor['suspend'].eq(True), 'value'] = np.nan
value_factor['value'] = value_factor.groupby('trade_date')['value'].transform(wz)
value_factor = value_factor[['trade_date', 'ts_code', 'value']]
style_factor_list.append(value_factor)


## 合并八个风格暴露

按 `trade_date + ts_code` 左连接各因子，保留统一股票面板的行，并按日期排序。


In [ ]:
for factor in style_factor_list:
    style_factor_exposure =pd.merge(left=style_factor_exposure, right=factor, on=["trade_date", "ts_code"], how="left")

style_factor_exposure = style_factor_exposure.sort_values('trade_date').reset_index(drop=True)

## 屏蔽停牌日暴露

合并停牌标记，将停牌日最终八个暴露全部置为缺失，再移除辅助状态列。


In [ ]:
style_factor_exposure = style_factor_exposure.merge(
    panel[['trade_date', 'ts_code', 'suspend']],
    on=['trade_date', 'ts_code'],
    how='left'
)

factor_cols = ['size', 'mom', 'vol1', 'vol2', 'liq1', 'liq2', 'beta', 'value']
style_factor_exposure.loc[style_factor_exposure['suspend'] == True, factor_cols] = np.nan

style_factor_exposure = style_factor_exposure.drop(columns=['suspend'])

## 预览结果

查看表头和表尾；保存列为日期、股票代码及八个标准化风格暴露。


In [ ]:
style_factor_exposure.head()

In [ ]:
style_factor_exposure.tail()

## 保存风格暴露

写入 `STYLE_FACTOR_OUTPUT`，下一步 Alpha 计算会读取该文件进行风格中性化。


In [ ]:
style_factor_exposure.to_parquet(STYLE_FACTOR_OUTPUT, index=False)